
## Clarified Objective and Methodology

This notebook performs an **exploratory analysis of inter-dataset visual similarity**.
A **single, common backbone** (ResNet-50 pre-trained on ImageNet) is used **without any fine-tuning**
to extract embeddings from images coming from different datasets (e.g., COCO, DOTA, DIOR).

All datasets are processed by the **same frozen network**, ensuring that the resulting embeddings
lie in a shared feature space and are therefore directly comparable.

Dimensionality reduction techniques (PCA and t-SNE) are used **only for visualization** purposes.
This analysis provides qualitative intuition about potential domain shifts, and does **not**
evaluate object detection performance.


In this study, we analyze the visual similarity between several image datasets by projecting them into a shared feature space using a common backbone network. Specifically, we employ a ResNet-50 model pre-trained on ImageNet as a fixed feature extractor to generate high-dimensional embeddings for images from different datasets, including COCO, DOTA, and DIOR. By using the same pre-trained backbone for all datasets, we ensure that the extracted representations are directly comparable and not biased by dataset-specific training. These embeddings are then reduced to a low-dimensional space using PCA followed by t-SNE, allowing for qualitative visualization of their distribution. This approach provides an exploratory analysis of inter-dataset similarity and offers intuition about potential domain shifts, which are known to impact cross-domain transfer performance in object detection tasks. While this analysis does not measure task performance directly, it serves as a complementary perspective to performance-based evaluations reported in the literature.

In [1]:
import torch, torchvision.transforms
from torchvision.models import resnet50, ResNet50_Weights

In [2]:
import requests
from PIL import Image

#test
'''url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = Image.open(requests.get(url, stream=True).raw)'''

'url = "http://images.cocodataset.org/val2017/000000039769.jpg"\nimage = Image.open(requests.get(url, stream=True).raw)'

## Dataset Loading from Hugging Face

Datasets are loaded directly from Hugging Face repositories.
This avoids manual downloads and ensures reproducibility.
Only image data is used; annotations are intentionally ignored.


In [3]:
from datasets import load_dataset

# Charger les datasets
#coco_ds = load_dataset("HichTala/coco-background", split="train")
#dota_ds = load_dataset("HichTala/dota-background", split="train")
dior_ds = load_dataset("HichTala/dior", split="train")


/opt/miniconda3/envs/image_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Common Backbone: ResNet-50 (ImageNet)

We use a ResNet-50 model pre-trained on ImageNet as a common backbone for all datasets.
The classification head is removed in order to extract global image embeddings.
The network is kept frozen and used only in inference mode.

In [4]:
model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)

In [5]:
from torchvision.models import ResNet50_Weights

weights = ResNet50_Weights.IMAGENET1K_V2
preprocess = weights.transforms()


## Device Selection

Computation is performed on the Apple MPS backend when available.
If MPS is not available, the CPU is used as a fallback.
This ensures compatibility across different hardware configurations.


In [6]:
model

if torch.backends.mps.is_available() and torch.backends.mps.is_built():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

model = model.to(device)

print("Using device:", device)


Using device: mps


## 3. Embedding Extraction Function

## Embedding Extraction Function

This function extracts global image embeddings by forwarding batches of images
through the frozen ResNet-50 backbone.
The output corresponds to the 2048-dimensional representation obtained after
global average pooling, before the classification layer.


In [7]:
import torch
import numpy as np
from tqdm import tqdm

@torch.no_grad()
def extract_embeddings(model, dataloader, device):
    """
    Extract image embeddings using a frozen ResNet-50 backbone.

    Args:
        model (torch.nn.Module): ResNet-50 with fc = Identity()
        dataloader (DataLoader): DataLoader returning (image, _)
        device (str): 'mps' or 'cpu'

    Returns:
        np.ndarray: Array of shape (N, 2048)
    """
    model.eval()
    embeddings = []

    for images, _ in tqdm(dataloader, desc="Extracting embeddings"):
        images = images.to(device)

        # Forward pass
        feats = model(images)   # shape: [B, 2048]

        embeddings.append(feats.cpu().numpy())

    embeddings = np.concatenate(embeddings, axis=0)
    return embeddings




## 4. Dataset Loading (Few-Shot)

## Few-Shot Sampling

A few-shot protocol is adopted by randomly sampling a limited number of images
from each dataset.
This simulates data-scarce scenarios and ensures a balanced comparison
between datasets of different sizes.


In [8]:
few_shot_n = 20

#coco_sample = coco_ds.shuffle(seed=42).select(range(few_shot_n))
#dota_sample = dota_ds.shuffle(seed=42).select(range(few_shot_n))
dior_sample = dior_ds.shuffle(seed=42).select(range(few_shot_n))

In [9]:
from PIL import Image
import numpy as np

def hf_dataset_to_pil(dataset):
    images = []
    for item in dataset:
        # Dans ces datasets, l'image est stockée dans la colonne "image"
        img = item["image"]
        if isinstance(img, Image.Image):
            images.append(img.convert("RGB"))
        else:
            images.append(Image.fromarray(img).convert("RGB"))
    return images

##coco_imgs = hf_dataset_to_pil(coco_sample)
##dota_imgs = hf_dataset_to_pil(dota_sample)
dior_imgs = hf_dataset_to_pil(dior_sample)


## Dataset Wrapper for PyTorch

A custom PyTorch Dataset is defined to convert the sampled images
into tensors and apply the required preprocessing.
Dummy labels are returned to comply with the DataLoader interface.


In [10]:
from torch.utils.data import DataLoader, Dataset

class FewShotListDataset(Dataset):
    def __init__(self, pil_images, transform):
        self.images = pil_images
        self.transform = transform
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        return self.transform(self.images[idx]), 0

def fewshot_loader(pil_list, batch_size=8):
    return DataLoader(FewShotListDataset(pil_list, preprocess), batch_size=batch_size)


## 9. DataLoader Construction

DataLoaders are created to efficiently batch images and iterate over
the datasets during embedding extraction.
Shuffling is disabled to preserve deterministic ordering.


In [11]:
batch_size = 8

dior_dataset = FewShotListDataset(dior_imgs, preprocess)
dior_loader = DataLoader(dior_dataset, batch_size=batch_size, shuffle=False)

print(len(dior_loader.dataset))


20


## 5. Embedding Extraction

## Embedding Extraction

Image embeddings are extracted for each dataset using the same frozen backbone.
Each image is mapped to a 2048-dimensional feature vector,
allowing direct comparison across datasets.


## Chosen Method

create_feature_extractor is useful for accessing intermediate feature maps, but for our analysis we only require the final global representation before classification, which is directly obtained by removing the classification head of the network.

In [12]:
#emb_coco = extract_embeddings(model, coco_loader, device)
#emb_dota = extract_embeddings(model, dota_loader, device)
emb_dior = extract_embeddings(model, dior_loader, device)

print(emb_dior.shape)


Extracting embeddings: 100%|██████████| 3/3 [00:00<00:00,  7.07it/s]

(20, 1000)


## Test Part

In [13]:
'''import torchvision

inputs = torchvision.transforms.ToTensor()(image)'''

'import torchvision\n\ninputs = torchvision.transforms.ToTensor()(image)'

In [14]:
'''from torch import nn

model.fc = nn.Linear(in_features=2048, out_features=2, bias=True)'''

'from torch import nn\n\nmodel.fc = nn.Linear(in_features=2048, out_features=2, bias=True)'

In [15]:
'''inputs = inputs.unsqueeze(0)'''

'inputs = inputs.unsqueeze(0)'

In [16]:
'''outputs = model(inputs)'''

'outputs = model(inputs)'

In [17]:
'''outputs.shape'''

'outputs.shape'

In [18]:
'''from torchvision.models.feature_extraction import create_feature_extractor

feature_extractor = create_feature_extractor(model, {'layer1': 'feat1', 'layer2': 'feat2', 'layer3': 'feat3', 'layer4': 'feat4'})'''

"from torchvision.models.feature_extraction import create_feature_extractor\n\nfeature_extractor = create_feature_extractor(model, {'layer1': 'feat1', 'layer2': 'feat2', 'layer3': 'feat3', 'layer4': 'feat4'})"

In [19]:
'''feature_extractor(inputs)['feat4'].shape'''

"feature_extractor(inputs)['feat4'].shape"

In [20]:
'''nn.Flatten()(feature_extractor(inputs)['feat4']).shape'''

"nn.Flatten()(feature_extractor(inputs)['feat4']).shape"

## 6. Dimensionality Reduction and Visualization

## Dimensionality Reduction

Principal Component Analysis (PCA) is first applied to reduce noise
and stabilize the embedding space.
The reduced embeddings are then projected to two dimensions using t-SNE
for visualization purposes.


In [ ]:
X = np.vstack([emb_dior])

labels = (
["DIOR"] * len(emb_dior)
)


: 

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

X_pca = PCA(n_components=15, random_state=0).fit_transform(X)

X_tsne = TSNE(
    n_components=2,
    perplexity=10,
    random_state=0,
    init="pca"
).fit_transform(X_pca)


In [ ]:
'''X = np.vstack([emb_coco, emb_dota, emb_dior])
labels = ["COCO"]*len(emb_coco) + ["DOTA"]*len(emb_dota) + ["DIOR"]*len(emb_dior)

X_pca = PCA(n_components=50, random_state=0).fit_transform(X)
X_tsne = TSNE(n_components=2, perplexity=30, random_state=0, init="pca").fit_transform(X_pca)'''

X = np.vstack([emb_dior])
labels =["DIOR"]*len(emb_dior)

X_pca = PCA(n_components=50, random_state=0).fit_transform(X)
X_tsne = TSNE(n_components=2, perplexity=30, random_state=0, init="pca").fit_transform(X_pca)

## Visualization

The 2D t-SNE embeddings are visualized to analyze the distribution,
overlap, and separation between datasets.
This qualitative analysis provides intuition about inter-dataset
visual similarity and potential domain shifts.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 6))
sns.scatterplot(x=X_tsne[:, 0], y=X_tsne[:, 1], hue=labels, alpha=0.7)
plt.title("t-SNE of Few-Shot Embeddings")
plt.show()


## Notes and Limitations

- This study is exploratory and qualitative in nature.
- No fine-tuning or task-specific training is performed.
- Detection performance metrics (e.g., mAP) are not evaluated.
- Observations should be interpreted as representation-level insights.
